In [7]:
import polars as pl
from pathlib import Path

from sqlalchemy import create_engine,text
from urllib.parse import quote_plus
from datetime import datetime, timezone
from uuid import uuid4
import re




In [8]:

connection_string = quote_plus(
    "DRIVER={ODBC Driver 18 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=netsuite;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={connection_string}"
)

In [9]:
#Create ID for Pipeline run in audit table
pipeline_run_id = str(uuid4())
pipeline_start_time = datetime.now()

with engine.begin() as connection:
    connection.execute(
        text("""
            INSERT INTO dbo.audit_pipeline_run
            (
                pipeline_run_id,
                pipeline_name,
                start_time,
                status
            )
            VALUES
            (
                :pipeline_run_id,
                :pipeline_name,
                :start_time,
                'RUNNING'
            )
        """),
        {
            "pipeline_run_id": pipeline_run_id,
            "pipeline_name": "netsuite_raw_ingestion",
            "start_time": pipeline_start_time,
        },
    )

In [14]:
# ------------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------------

directory_path = Path(
    "D:/Nazeer Joseph Data Universe/Data Load/Netsuite ETL Full Project/data"
)

raw_tables = {
    "customer": "dbo.raw_customer",
    "item_category": "dbo.raw_item_category",
    "item_pattern": "dbo.raw_item_pattern",
    "item": "dbo.raw_item",
    "subsidiary": "dbo.raw_subsidiary",
    "sales_budget": "dbo.raw_sales_budget",
    "transaction": "dbo.raw_transaction",
    "transactionline": "dbo.raw_transactionline",
    "fx_avg_rate": "dbo.raw_fx_avg_rate",
    "deleted_records": "dbo.raw_deleted_records",
    "user_rls": "dbo.raw_user_rls",
}


# ------------------------------------------------------------------
# CREATE PIPELINE RUN
# ------------------------------------------------------------------

pipeline_run_id = str(uuid4())
pipeline_start_time = datetime.now()
pipeline_name = "netsuite_raw_ingestion"


# Insert one audit record for the complete pipeline run.
with engine.begin() as connection:
    connection.execute(
        text("""
            INSERT INTO dbo.audit_pipeline_run
            (
                pipeline_run_id,
                pipeline_name,
                start_time,
                status
            )
            VALUES
            (
                :pipeline_run_id,
                :pipeline_name,
                :start_time,
                'RUNNING'
            )
        """),
        {
            "pipeline_run_id": pipeline_run_id,
            "pipeline_name": pipeline_name,
            "start_time": pipeline_start_time,
        },
    )


# Optional dictionary for inspecting DataFrames while developing.
dataframes = {}


# ------------------------------------------------------------------
# LOAD RAW FILES
# ------------------------------------------------------------------

try:

    for file_path in directory_path.glob("*.csv"):

        file_name = file_path.stem.lower()

        # Ignore files that are not part of the pipeline.
        if file_name not in raw_tables:
            print(f"Skipped: {file_path.name}")
            continue

        table_start_time = datetime.now()
        table_name = raw_tables[file_name]

        # Read the original CSV.
        df = pl.read_csv(
            file_path,
            try_parse_dates=True,
            separator=";",
        )

        # Store the number of physical rows in the original file.
        # For FX, the loaded count may be higher after unpivoting.
        source_row_count = df.height


        # ----------------------------------------------------------
        # FX RATE TRANSFORMATION
        # ----------------------------------------------------------

        if file_name == "fx_avg_rate":

            # Dynamically identify every column whose name is a date.
            # This works whether the file contains 1, 3 or many months.
            date_columns = [
                column
                for column in df.columns
                if re.fullmatch(r"\d{2}/\d{2}/\d{4}", column.strip())
            ]

            if not date_columns:
                raise ValueError("No FX date columns were found in fx_avg_rate.csv")

            df = (
                df.unpivot(
                    index=["original_currency", "target_currency"],
                    on=date_columns,
                    variable_name="rate_date",
                    value_name="avg_rate",
                )
                .with_columns(
                    pl.col("rate_date").str.strptime(
                        pl.Date,
                        format="%d/%m/%Y",
                        strict=True,
                    ),
                    pl.col("avg_rate").cast(
                        pl.Float64,
                        strict=False,
                    ),
                )
            )

        # ----------------------------------------------------------
        # ADD INGESTION METADATA TO DATAFRAMES
        # ----------------------------------------------------------

        ingestion_timestamp = datetime.now()

        df = df.with_columns(
            pl.lit(pipeline_run_id).alias("pipeline_run_id"),
            pl.lit(ingestion_timestamp).alias(
                "ingestion_timestamp"
            ),
            pl.lit(file_path.name).alias("source_file_name"),
        )

        dataframes[file_name] = df


        # ----------------------------------------------------------
        # LOAD TABLE AND WRITE AUDIT RECORD
        # ----------------------------------------------------------

        # Everything inside this block is committed together.
        # If the insert or audit fails, the table transaction rolls back.
        with engine.begin() as connection:

            # The CSV files represent the latest raw snapshot.
            # Keep the table structure but remove the previous data.
            connection.execute(
                text(f"DELETE FROM {table_name}")
            )

            # Insert the new raw data.
            df.write_database(
                table_name=table_name,
                connection=connection,
                if_table_exists="append",
            )

            # Count the records loaded by this pipeline run.
            loaded_row_count = connection.execute(
                text(f"""
                    SELECT COUNT(*)
                    FROM {table_name}
                    WHERE pipeline_run_id = :pipeline_run_id
                """),
                {
                    "pipeline_run_id": pipeline_run_id,
                },
            ).scalar_one()

            # Ensure the final DataFrame count matches SQL Server.
            if loaded_row_count != df.height:
                raise ValueError(
                    f"Row-count mismatch for {table_name}. "
                    f"Expected {df.height}, loaded {loaded_row_count}."
                )

            # Write one successful audit record for this table.
            connection.execute(
                text("""
                    INSERT INTO dbo.audit_table_load
                    (
                        pipeline_run_id,
                        table_name,
                        source_file_name,
                        source_row_count,
                        loaded_row_count,
                        start_time,
                        end_time,
                        status
                    )
                    VALUES
                    (
                        :pipeline_run_id,
                        :table_name,
                        :source_file_name,
                        :source_row_count,
                        :loaded_row_count,
                        :start_time,
                        :end_time,
                        'SUCCESS'
                    )
                """),
                {
                    "pipeline_run_id": pipeline_run_id,
                    "table_name": table_name,
                    "source_file_name": file_path.name,
                    "source_row_count": source_row_count,
                    "loaded_row_count": loaded_row_count,
                    "start_time": table_start_time,
                    "end_time": datetime.now(),
                },
            )

        print(
            f"Loaded {file_path.name} into {table_name}: "
            f"{loaded_row_count} rows"
        )


    # --------------------------------------------------------------
    # MARK PIPELINE AS SUCCESSFUL
    # --------------------------------------------------------------

    with engine.begin() as connection:
        connection.execute(
            text("""
                UPDATE dbo.audit_pipeline_run
                SET
                    end_time = :end_time,
                    status = 'SUCCESS'
                WHERE pipeline_run_id = :pipeline_run_id
            """),
            {
                "pipeline_run_id": pipeline_run_id,
                "end_time": datetime.now(),
            },
        )

    print(
        f"Pipeline completed successfully: {pipeline_run_id}"
    )

# ------------------------------------------------------------------
# PIPELINE FAILURE HANDLING
# ------------------------------------------------------------------

except Exception as error:

    with engine.begin() as connection:
        connection.execute(
            text("""
                UPDATE dbo.audit_pipeline_run
                SET
                    end_time = :end_time,
                    status = 'FAILED',
                    error_message = :error_message
                WHERE pipeline_run_id = :pipeline_run_id
            """),
            {
                "pipeline_run_id": pipeline_run_id,
                "end_time": datetime.now(),
                "error_message": str(error),
            },
        )

    print(f"Pipeline failed: {error}")

    # Raise the error again so Python does not hide the failure.
    raise

Loaded customer.csv into dbo.raw_customer: 10000 rows
Loaded deleted_records.csv into dbo.raw_deleted_records: 1 rows
Loaded fx_avg_rate.csv into dbo.raw_fx_avg_rate: 102 rows
Loaded item.csv into dbo.raw_item: 20000 rows
Loaded item_category.csv into dbo.raw_item_category: 50 rows
Loaded item_pattern.csv into dbo.raw_item_pattern: 6 rows
Loaded sales_budget.csv into dbo.raw_sales_budget: 780 rows
Loaded subsidiary.csv into dbo.raw_subsidiary: 120 rows
Loaded transaction.csv into dbo.raw_transaction: 500000 rows
Loaded transactionline.csv into dbo.raw_transactionline: 1000000 rows
Loaded user_rls.csv into dbo.raw_user_rls: 11 rows
Pipeline completed successfully: c27eb047-7366-4d7f-9fc4-4dea45f7d392


In [13]:
from sqlalchemy import text

queries = {
    "Total rows": """
        SELECT COUNT(*) AS row_count
        FROM dbo.stg_customer;
    """,
    "NULL primary keys": """
        SELECT COUNT(*) AS null_customer_keys
        FROM dbo.stg_customer
        WHERE customer_nsid IS NULL;
    """,
    "Duplicate primary keys": """
        SELECT
            customer_nsid,
            COUNT(*) AS duplicate_count
        FROM dbo.stg_customer
        GROUP BY customer_nsid
        HAVING COUNT(*) > 1;
    """,
}

with engine.connect() as connection:
    for label, query in queries.items():
        print(f"\n{label}")

        result = connection.execute(text(query))

        for row in result:
            print(row)


Total rows
(10000,)

NULL primary keys
(0,)

Duplicate primary keys


In [19]:
from sqlalchemy import text

with engine.connect() as connection:
    result = connection.execute(
        text("""
            SELECT DISTINCT table_name
            FROM INFORMATION_SCHEMA.COLUMNS
            WHERE table_name LIKE 'stg_%'
            ORDER BY table_name
        """)
    )

    stg_views = result.scalars().all()

print(stg_views)

['stg_customer', 'stg_deleted_records', 'stg_fx_avg_rate', 'stg_item', 'stg_item_category', 'stg_item_pattern', 'stg_sales_budget', 'stg_subsidiary', 'stg_transaction', 'stg_transactionline']


In [24]:
primary_keys = {
    "stg_customer": ["customer_nsid"],
    "stg_deleted_records": ["transaction_nsid"],
    "stg_fx_avg_rate": [
        "original_currency",
        "target_currency",
        "rate_date",
    ],
    "stg_item": ["item_nsid"],
    "stg_item_category": ["item_category_nsid"],
    "stg_item_pattern": ["item_pattern_nsid"],
    "stg_sales_budget": ["sales_budget_id"],
    "stg_subsidiary": ["bu_nsid"],
    "stg_transaction": ["transaction_nsid"],
    "stg_transactionline": [
        "transaction_nsid",
        "transaction_line_nsid",
    ],
}
results = []

with engine.connect() as connection:
    stg_views = connection.execute(
        text("""
            SELECT DISTINCT table_name
            FROM INFORMATION_SCHEMA.COLUMNS
            WHERE table_schema = 'dbo'
              AND table_name LIKE 'stg_%'
            ORDER BY table_name
        """)
    ).scalars().all()

    for table_name in stg_views:
        keys = primary_keys.get(table_name)

        if not keys:
            print(f"No key configured for {table_name}")
            continue

        qualified_table = f"dbo.[{table_name}]"

        key_columns = ", ".join(f"[{column}]" for column in keys)

        null_condition = " OR ".join(
            f"[{column}] IS NULL" for column in keys
        )

        row_count = connection.execute(
            text(f"SELECT COUNT(*) FROM {qualified_table}")
        ).scalar_one()

        null_key_count = connection.execute(
            text(f"""
                SELECT COUNT(*)
                FROM {qualified_table}
                WHERE {null_condition}
            """)
        ).scalar_one()

        duplicates = connection.execute(
            text(f"""
                SELECT
                    {key_columns},
                    COUNT(*) AS duplicate_count
                FROM {qualified_table}
                GROUP BY {key_columns}
                HAVING COUNT(*) > 1
            """)
        ).mappings().all()

        results.append({
            "table_name": table_name,
            "primary_key": keys,
            "row_count": row_count,
            "null_key_count": null_key_count,
            "duplicate_groups": len(duplicates),
            "duplicates": duplicates,
        })

for result in results:
    print(f"\nTable: {result['table_name']}")
    print(f"Key: {', '.join(result['primary_key'])}")
    print(f"Rows: {result['row_count']}")
    print(f"NULL keys: {result['null_key_count']}")
    print(f"Duplicate groups: {result['duplicate_groups']}")

    for duplicate in result["duplicates"]:
        print(dict(duplicate))


Table: stg_customer
Key: customer_nsid
Rows: 10000
NULL keys: 0
Duplicate groups: 0

Table: stg_deleted_records
Key: transaction_nsid
Rows: 1
NULL keys: 0
Duplicate groups: 0

Table: stg_fx_avg_rate
Key: original_currency, target_currency, rate_date
Rows: 102
NULL keys: 0
Duplicate groups: 0

Table: stg_item
Key: item_nsid
Rows: 20000
NULL keys: 0
Duplicate groups: 0

Table: stg_item_category
Key: item_category_nsid
Rows: 50
NULL keys: 0
Duplicate groups: 0

Table: stg_item_pattern
Key: item_pattern_nsid
Rows: 6
NULL keys: 0
Duplicate groups: 0

Table: stg_sales_budget
Key: sales_budget_id
Rows: 780
NULL keys: 0
Duplicate groups: 0

Table: stg_subsidiary
Key: bu_nsid
Rows: 120
NULL keys: 0
Duplicate groups: 0

Table: stg_transaction
Key: transaction_nsid
Rows: 500000
NULL keys: 0
Duplicate groups: 0

Table: stg_transactionline
Key: transaction_nsid, transaction_line_nsid
Rows: 1000000
NULL keys: 0
Duplicate groups: 0


In [42]:
from sqlalchemy import text
from sqlalchemy.engine import Engine


RELATIONSHIP_CHECKS = {
    "Transaction lines missing transactions": """
        SELECT COUNT(*)
        FROM dbo.stg_transactionline AS tl
        LEFT JOIN dbo.stg_transaction AS t
            ON tl.transaction_nsid = t.transaction_nsid
        WHERE tl.transaction_nsid IS NOT NULL
          AND t.transaction_nsid IS NULL;
    """,

    "Transaction lines missing items": """
        SELECT COUNT(*)
        FROM dbo.stg_transactionline AS tl
        LEFT JOIN dbo.stg_item AS i
            ON tl.item_nsid = i.item_nsid
        WHERE tl.item_nsid IS NOT NULL
          AND i.item_nsid IS NULL;
    """,
    
    "Transactions missing customers": """
    SELECT COUNT(*)
    FROM dbo.stg_transaction AS t
    LEFT JOIN dbo.stg_customer AS c
        ON t.customer_nsid = c.customer_nsid
    WHERE t.customer_nsid IS NOT NULL
      AND c.customer_nsid IS NULL;
    """,
    
    "Items missing item categories": """
        SELECT COUNT(*)
        FROM dbo.stg_item AS i
        LEFT JOIN dbo.stg_item_category AS c
            ON i.item_category_nsid = c.item_category_nsid
        WHERE i.item_category_nsid IS NOT NULL
          AND c.item_category_nsid IS NULL;
    """,

    "Items missing item patterns": """
        SELECT COUNT(*)
        FROM dbo.stg_item AS i
        LEFT JOIN dbo.stg_item_pattern AS p
            ON i.item_pattern_nsid = p.item_pattern_nsid
        WHERE i.item_pattern_nsid IS NOT NULL
          AND p.item_pattern_nsid IS NULL;
    """,

    "Transactions missing subsidiaries": """
        SELECT COUNT(*)
        FROM dbo.stg_transaction AS t
        LEFT JOIN dbo.stg_subsidiary AS s
            ON t.bu_nsid = s.bu_nsid
        WHERE t.bu_nsid IS NOT NULL
          AND s.bu_nsid IS NULL;
    """,

    "Sales budgets missing subsidiaries": """
        SELECT COUNT(*)
        FROM dbo.stg_sales_budget AS b
        LEFT JOIN dbo.stg_subsidiary AS s
            ON b.bu_code = s.bu_code
        WHERE b.bu_code IS NOT NULL
          AND s.bu_code IS NULL;
    """,

    "Transaction currencies missing USD FX rates": """
        SELECT COUNT(*)
        FROM (
            SELECT DISTINCT foreign_currency
            FROM dbo.stg_transactionline
            WHERE foreign_currency IS NOT NULL
        ) AS currencies
        LEFT JOIN (
            SELECT DISTINCT original_currency
            FROM dbo.stg_fx_avg_rate
            WHERE target_currency = 'USD'
        ) AS fx
            ON currencies.foreign_currency = fx.original_currency
        WHERE fx.original_currency IS NULL;
    """,

    "Budget currencies missing USD FX rates": """
        SELECT COUNT(*)
        FROM (
            SELECT DISTINCT bu_currency
            FROM dbo.stg_sales_budget
            WHERE bu_currency IS NOT NULL
        ) AS currencies
        LEFT JOIN (
            SELECT DISTINCT original_currency
            FROM dbo.stg_fx_avg_rate
            WHERE target_currency = 'USD'
        ) AS fx
            ON currencies.bu_currency = fx.original_currency
        WHERE fx.original_currency IS NULL;
    """,
}


FX_DETAIL_CHECKS = {
    "Transaction currencies missing USD FX rates": """
        SELECT DISTINCT
            tl.foreign_currency
        FROM dbo.stg_transactionline AS tl
        LEFT JOIN dbo.stg_fx_avg_rate AS fx
            ON tl.foreign_currency = fx.original_currency
           AND fx.target_currency = 'USD'
        WHERE tl.foreign_currency IS NOT NULL
          AND fx.original_currency IS NULL
        ORDER BY tl.foreign_currency;
    """,

    "Budget currencies missing USD FX rates": """
        SELECT DISTINCT
            b.bu_currency
        FROM dbo.stg_sales_budget AS b
        LEFT JOIN dbo.stg_fx_avg_rate AS fx
            ON b.bu_currency = fx.original_currency
           AND fx.target_currency = 'USD'
        WHERE b.bu_currency IS NOT NULL
          AND fx.original_currency IS NULL
        ORDER BY b.bu_currency;
    """,
}


def run_relationship_checks(
    engine: Engine,
) -> tuple[dict[str, int], dict[str, list[str]]]:
    """Run relationship checks and retrieve details for FX failures."""

    results: dict[str, int] = {}
    fx_details: dict[str, list[str]] = {}

    with engine.connect() as connection:
        for check_name, query in RELATIONSHIP_CHECKS.items():
            count = connection.execute(text(query)).scalar_one()
            results[check_name] = count

            if count > 0 and check_name in FX_DETAIL_CHECKS:
                missing_currencies = connection.execute(
                    text(FX_DETAIL_CHECKS[check_name])
                ).scalars().all()

                fx_details[check_name] = list(missing_currencies)

    return results, fx_details


def print_relationship_check_results(
    results: dict[str, int],
    fx_details: dict[str, list[str]],
) -> None:
    """Print relationship-check results and FX failure details."""

    print("Relationship check results")
    print("-" * 60)

    for check_name, count in results.items():
        status = "PASS" if count == 0 else "FAIL"
        print(f"[{status}] {check_name}: {count:,}")

        if check_name in fx_details:
            missing = ", ".join(fx_details[check_name])
            print(f"       Missing currencies: {missing}")


relationship_results, fx_missing_details = run_relationship_checks(engine)

print_relationship_check_results(
    relationship_results,
    fx_missing_details,
)

Relationship check results
------------------------------------------------------------
[PASS] Transaction lines missing transactions: 0
[PASS] Transaction lines missing items: 0
[PASS] Transactions missing customers: 0
[PASS] Items missing item categories: 0
[PASS] Items missing item patterns: 0
[PASS] Transactions missing subsidiaries: 0
[PASS] Sales budgets missing subsidiaries: 0
[FAIL] Transaction currencies missing USD FX rates: 1
       Missing currencies: USD
[FAIL] Budget currencies missing USD FX rates: 1
       Missing currencies: USD
